In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

## 1、下载数据集

In [3]:
#sum_datasets = load_dataset("hugcyp/LCSTS", cache_dir="../data",trust_remote_code=True)

In [4]:
#sum_datasets

In [5]:
sum_datasets = load_dataset("hugcyp/LCSTS",split="train[:50000]")

In [6]:
sum_datasets

Dataset({
    features: ['summary', 'text'],
    num_rows: 50000
})

## 2、数据预处理

In [7]:
sum_datasets = sum_datasets.train_test_split(test_size=0.2, seed=42)
sum_datasets

DatasetDict({
    train: Dataset({
        features: ['summary', 'text'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['summary', 'text'],
        num_rows: 10000
    })
})

In [8]:
sum_datasets["train"][0]

{'summary': '90后12万入股市2天3部iphone6没了！',
 'text': '在北京工作刚刚25岁的张涛是一位今年入市的新股民，一入市，就“歪打正着”接触了分级基金。在12月初的一次出差间隙，他专程回杭州说服父母拿出50万元“养老钱”给他炒股。“短短两天，就亏了三部iphone6。”张涛惊慌失措。'}

In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("lemon234071/t5-base-Chinese", cache_dir='./models')


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\convert_slow_tokenizer.py:551: UserWarning: The sentencepiece

In [10]:
# tokenizer = AutoTokenizer.from_pretrained("uer/t5-base-chinese-cluecorpussmall", cache_dir='./models')
# tokenizer

In [11]:
def process_func(datasets):
    contents = ["摘要生成: \n" + e for e in datasets["text"]]
    inputs = tokenizer(contents, max_length=512, truncation=True)
    labels = tokenizer(text_target=datasets["summary"], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

In [12]:
tokenized_ds = sum_datasets.map(process_func, batched=True)
tokenized_ds

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['summary', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['summary', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
})

In [13]:
print(tokenized_ds["train"][0])

{'summary': '90后12万入股市2天3部iphone6没了！', 'text': '在北京工作刚刚25岁的张涛是一位今年入市的新股民，一入市，就“歪打正着”接触了分级基金。在12月初的一次出差间隙，他专程回杭州说服父母拿出50万元“养老钱”给他炒股。“短短两天，就亏了三部iphone6。”张涛惊慌失措。', 'input_ids': [259, 23334, 16620, 267, 259, 22132, 2190, 20396, 903, 18254, 2724, 19292, 829, 13275, 5009, 1647, 996, 14991, 4165, 2713, 261, 766, 1647, 996, 261, 1511, 469, 23752, 1946, 2289, 1493, 340, 14911, 747, 814, 3817, 8416, 299, 666, 667, 26332, 422, 7155, 1083, 4074, 4396, 25929, 261, 1456, 4692, 4120, 1496, 11762, 1893, 3657, 15631, 28937, 954, 7748, 469, 24046, 3005, 340, 28469, 13315, 4165, 6088, 4712, 4712, 29340, 261, 1511, 22308, 747, 991, 1693, 4986, 394, 3025, 2724, 19292, 8971, 18101, 3915, 24617, 299, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1089, 1412, 667, 

In [14]:
tokenizer.decode(tokenized_ds["train"][0]["input_ids"])

'摘要生成: 在北京工作刚刚25岁的张涛是一位今年入市的新股民,一入市,就“歪打正着”接触了分级基金。在12月初的一次出差间隙,他专程回杭州说服父母拿出50万元“养老钱”给他炒股。“短短两天,就亏了三部iphone6。”张涛惊慌失措。</s>'

In [15]:
tokenizer.decode(tokenized_ds["train"][0]["labels"])

'90后12万入股市2天3部iphone6没了!</s>'

## 3、创建模型

In [16]:

model = AutoModelForSeq2SeqLM.from_pretrained("lemon234071/t5-base-Chinese", cache_dir='./models')

d:\program\anaconda3\envs\transformers\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


## 4、创建评估函数

In [17]:
#!pip install rouge_chinese

In [18]:
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()

def compute_metric(evalPred):
    predictions, labels = evalPred
    decode_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decode_preds = [" ".join(p) for p in decode_preds]
    decode_labels = [" ".join(l) for l in decode_labels]
    scores = rouge.get_scores(decode_preds, decode_labels, avg=True)
    return {
        "rouge-1": scores["rouge-1"]["f"],
        "rouge-2": scores["rouge-2"]["f"],
        "rouge-l": scores["rouge-l"]["f"],
    }

## 5、创建训练器

In [19]:
# train_args = Seq2SeqTrainingArguments(
#     output_dir="./summary",
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     #gradient_accumulation_steps=2,
#     logging_steps=10,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     metric_for_best_model="rouge-l",
#     predict_with_generate=True
# )

In [20]:
train_args = Seq2SeqTrainingArguments(
    output_dir="./summary",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=300,
    save_strategy="epoch",
    metric_for_best_model="rouge-l",
    predict_with_generate=True,
    report_to=['tensorboard']
)

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## 6、开始训练

In [21]:
trainer = Seq2SeqTrainer(
    args=train_args,
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metric,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer)
)

In [22]:
trainer.train()

  0%|          | 0/3750 [00:00<?, ?it/s]

{'loss': 11.0027, 'grad_norm': 939.9522094726562, 'learning_rate': 4.933333333333334e-05, 'epoch': 0.04}
{'loss': 6.4986, 'grad_norm': 15.527998924255371, 'learning_rate': 4.866666666666667e-05, 'epoch': 0.08}
{'loss': 4.7836, 'grad_norm': 6.35989236831665, 'learning_rate': 4.8e-05, 'epoch': 0.12}
{'loss': 4.4957, 'grad_norm': 16.320810317993164, 'learning_rate': 4.7333333333333336e-05, 'epoch': 0.16}
{'loss': 4.3774, 'grad_norm': 14.490204811096191, 'learning_rate': 4.666666666666667e-05, 'epoch': 0.2}
{'loss': 4.2363, 'grad_norm': 7.419302940368652, 'learning_rate': 4.600000000000001e-05, 'epoch': 0.24}


d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\generation\utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 3.423739433288574, 'eval_rouge-1': 0.34028274343583564, 'eval_rouge-2': 0.2030482409226334, 'eval_rouge-l': 0.30022441324587534, 'eval_runtime': 134.2272, 'eval_samples_per_second': 74.501, 'eval_steps_per_second': 1.17, 'epoch': 0.24}
{'loss': 4.1939, 'grad_norm': 18.314430236816406, 'learning_rate': 4.5333333333333335e-05, 'epoch': 0.28}
{'loss': 4.0655, 'grad_norm': 5.38582181930542, 'learning_rate': 4.466666666666667e-05, 'epoch': 0.32}
{'loss': 4.0737, 'grad_norm': 10.309065818786621, 'learning_rate': 4.4000000000000006e-05, 'epoch': 0.36}
{'loss': 4.0465, 'grad_norm': 16.519685745239258, 'learning_rate': 4.3333333333333334e-05, 'epoch': 0.4}
{'loss': 3.9935, 'grad_norm': 4.824881553649902, 'learning_rate': 4.266666666666667e-05, 'epoch': 0.44}
{'loss': 3.9329, 'grad_norm': 3.1898510456085205, 'learning_rate': 4.2e-05, 'epoch': 0.48}


...

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 2.9685792922973633, 'eval_rouge-1': 0.37760465025512197, 'eval_rouge-2': 0.2334750864851575, 'eval_rouge-l': 0.3348315671990117, 'eval_runtime': 188.9073, 'eval_samples_per_second': 52.936, 'eval_steps_per_second': 0.831, 'epoch': 2.64}
{'loss': 3.3986, 'grad_norm': 3.0244979858398438, 'learning_rate': 5.333333333333334e-06, 'epoch': 2.68}
{'loss': 3.3656, 'grad_norm': 2.738480806350708, 'learning_rate': 4.666666666666667e-06, 'epoch': 2.72}
{'loss': 3.3816, 'grad_norm': 3.0024828910827637, 'learning_rate': 4.000000000000001e-06, 'epoch': 2.76}
{'loss': 3.4208, 'grad_norm': 13.218899726867676, 'learning_rate': 3.3333333333333333e-06, 'epoch': 2.8}
{'loss': 3.3914, 'grad_norm': 4.201996803283691, 'learning_rate': 2.666666666666667e-06, 'epoch': 2.84}
{'loss': 3.3421, 'grad_norm': 2.844541072845459, 'learning_rate': 2.0000000000000003e-06, 'epoch': 2.88}


  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 2.96174955368042, 'eval_rouge-1': 0.3775228477774149, 'eval_rouge-2': 0.23366305211825192, 'eval_rouge-l': 0.3347607987243306, 'eval_runtime': 193.2762, 'eval_samples_per_second': 51.739, 'eval_steps_per_second': 0.812, 'epoch': 2.88}
{'loss': 3.4422, 'grad_norm': 2.8556602001190186, 'learning_rate': 1.3333333333333334e-06, 'epoch': 2.92}
{'loss': 3.4084, 'grad_norm': 3.246020555496216, 'learning_rate': 6.666666666666667e-07, 'epoch': 2.96}
{'loss': 3.3284, 'grad_norm': 2.8120195865631104, 'learning_rate': 0.0, 'epoch': 3.0}
{'train_runtime': 4562.2836, 'train_samples_per_second': 26.303, 'train_steps_per_second': 0.822, 'train_loss': 3.760310282389323, 'epoch': 3.0}


TrainOutput(global_step=3750, training_loss=3.760310282389323, metrics={'train_runtime': 4562.2836, 'train_samples_per_second': 26.303, 'train_steps_per_second': 0.822, 'total_flos': 1.5816917082636288e+16, 'train_loss': 3.760310282389323, 'epoch': 3.0})

## 7、评估

In [23]:
trainer.evaluate()

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\generation\utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 2.9646284580230713,
 'eval_rouge-1': 0.3779141772201781,
 'eval_rouge-2': 0.23399135945117372,
 'eval_rouge-l': 0.33521332768125855,
 'eval_runtime': 186.9791,
 'eval_samples_per_second': 53.482,
 'eval_steps_per_second': 0.84,
 'epoch': 3.0}

## 8、预测

In [24]:
from transformers import pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

In [54]:
text = '艾美百分百艺术家萨姆.萨莫雷（SamSamore）除了是一名摄影师，也是知名作家，他对传统故事及神话作品作了颠覆性改编。他为艾美酒店创作的神话作品在“迈阿密之夜”活动上发布。当然，他也加入了“开启艺术之门”计划，为艾美酒店设计房卡。'

In [55]:
pipe("生成摘要:\n" + text, max_length=64, do_sample=True)

[{'generated_text': '艾美百分百艺术家萨姆萨莫雷'}]